# Polish Companies Bankruptcy Prediction
## 1 Data Understanding

The objective of the project is to predict whether a company will go bankrupt within the next year using historical financial ratios.

### Dataset

Source: UCI Machine Learning Repository — Polish Companies Bankruptcy Dataset

For the initial modeling task, `5year.arff` is used. Despite its filename, this subset contains financial ratios from the 5th year of the forecasting period, with the class label indicating bankruptcy status 1 year later. 

The remaining ARFF subsets represent different forecasting horizons and are intentionally not combined with the 1-year prediction task.

Target:
- `0` = Non-bankrupt
- `1` = Bankrupt

In [1]:
# Import the needed libraries

from pathlib import Path
import numpy as np
import pandas as pd
from scipy.io import arff

In [2]:
#Create a data path

DATA_PATH = Path("../data/raw/5year.arff")

print(f"Dataset path: {DATA_PATH}")
print(f"File exists: {DATA_PATH.exists()}")

Dataset path: ..\data\raw\5year.arff
File exists: True


In [3]:
#  Convert arff file to DataFrame

raw_data, metadata = arff.loadarff(DATA_PATH)

bankruptcy_1y = pd.DataFrame(raw_data)

bankruptcy_1y.head()

,Attr1,Attr2,Attr3,Attr4,Attr5,Attr6,Attr7,Attr8,Attr9,Attr10,...,Attr56,Attr57,Attr58,Attr59,Attr60,Attr61,Attr62,Attr63,Attr64,class
0,0.088238,0.55472,0.01134,1.0205,-66.5200,0.342040,0.109490,0.57752,1.0881,0.32036,...,0.080955,0.275430,0.91905,0.002024,7.2711,4.7343,142.760,2.5568,3.2597,b'0'
1,-0.006202,0.48465,0.23298,1.5998,6.1825,0.000000,-0.006202,1.06340,1.2757,0.51535,...,-0.028591,-0.012035,1.00470,0.152220,6.0911,3.2749,111.140,3.2841,3.3700,b'0'
2,0.130240,0.22142,0.57751,3.6082,120.0400,0.187640,0.162120,3.05900,1.1415,0.67731,...,0.123960,0.192290,0.87604,0.000000,8.7934,2.9870,71.531,5.1027,5.6188,b'0'
3,-0.089951,0.88700,0.26927,1.5222,-55.9920,-0.073957,-0.089951,0.12740,1.2754,0.11300,...,0.418840,-0.796020,0.59074,2.878700,7.6524,3.3302,147.560,2.4735,5.9299,b'0'
4,0.048179,0.55041,0.10765,1.2437,-22.9590,0.000000,0.059280,0.81682,1.5150,0.44959,...,0.240400,0.107160,0.77048,0.139380,10.1180,4.0950,106.430,3.4294,3.3622,b'0'


In [4]:
#Check the size of rows and columns

print(f"Rows: {bankruptcy_1y.shape[0]:,}")
print(f"Columns: {bankruptcy_1y.shape[1]}")

Rows: 5,910
Columns: 65


In [5]:
bankruptcy_1y.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5910 entries, 0 to 5909
Data columns (total 65 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Attr1   5907 non-null   float64
 1   Attr2   5907 non-null   float64
 2   Attr3   5907 non-null   float64
 3   Attr4   5889 non-null   float64
 4   Attr5   5899 non-null   float64
 5   Attr6   5907 non-null   float64
 6   Attr7   5907 non-null   float64
 7   Attr8   5892 non-null   float64
 8   Attr9   5909 non-null   float64
 9   Attr10  5907 non-null   float64
 10  Attr11  5907 non-null   float64
 11  Attr12  5889 non-null   float64
 12  Attr13  5910 non-null   float64
 13  Attr14  5907 non-null   float64
 14  Attr15  5904 non-null   float64
 15  Attr16  5892 non-null   float64
 16  Attr17  5892 non-null   float64
 17  Attr18  5907 non-null   float64
 18  Attr19  5910 non-null   float64
 19  Attr20  5910 non-null   float64
 20  Attr21  5807 non-null   float64
 21  Attr22  5907 non-null   float64
 22  

In [6]:
# Convert the Target to integer safely

bankruptcy_1y["class"] = bankruptcy_1y["class"].apply(lambda x: x.decode("utf-8") if isinstance(x,bytes) else x).astype(int)

print("Target dtype:", bankruptcy_1y["class"].dtype)

bankruptcy_1y["class"].value_counts()

Target dtype: int64


class
0    5500
1     410
Name: count, dtype: int64

In [7]:
# Calculate the count and percentage of bankrupt and non bankrupt companies

target_distribution = (bankruptcy_1y["class"].value_counts().rename_axis("bankrupt").to_frame("count"))

target_distribution["percentage"] = (target_distribution["count"] / len(bankruptcy_1y) * 100).round(2)

target_distribution

,count,percentage
bankrupt,,
0,5500,93.06
1,410,6.94


In [8]:
#summarize the missing values by feature and rank the affected columns

missing_summary = pd.DataFrame({
    "missing_count": bankruptcy_1y.isna().sum(),
    "missing_percentage": (bankruptcy_1y.isna().mean() * 100).round(2)
})

missing_summary = (missing_summary.query("missing_count > 0").sort_values("missing_percentage", ascending=False))

missing_summary.head(15)

,missing_count,missing_percentage
Attr37,2548,43.11
Attr27,391,6.62
Attr60,268,4.53
Attr45,268,4.53
Attr24,135,2.28
Attr28,107,1.81
Attr64,107,1.81
Attr53,107,1.81
Attr54,107,1.81
Attr21,103,1.74


In [9]:
# Sum the total number of missing cells across the entire dataset
print(f"total missing values: {bankruptcy_1y.isna().sum().sum():,}")

total missing values: 4,666


## Data Quality Checks

The dataset is checked for:

- duplicate records,
- missing target values,
- unexpected target classes,
- infinite numerical values,
- constant or low-variability features,
- unusually large or small financial ratios.

In [10]:
# Check whether the dataset contains fully duplicated rows

duplicate_count = bankruptcy_1y.duplicated().sum()

print(f"Duplicate rows: {duplicate_count}")

Duplicate rows: 60


In [11]:
# Verify the target contains no missing values and only expected bankruptcy classes 0 and 1

target_missing = bankruptcy_1y["class"].isna().sum()
target_classes = bankruptcy_1y["class"].dropna().unique()

print(f"Missing target values: {target_missing:,}")
print(f"Target classes: {target_classes}")

Missing target values: 0
Target classes: [0 1]


In [12]:
# Check numerical features for positive and negative infinity.

feature_data = bankruptcy_1y.drop(columns="class")

infinite_counts = pd.Series(np.isinf(feature_data).sum(), index= feature_data.columns, name = "infinite_count")

infinite_counts = infinite_counts[infinite_counts > 0].sort_values(ascending=False)

print(f"Total infinite values: {infinite_counts.sum():,}")
infinite_counts.head(10)

Total infinite values: 0


Series([], Name: infinite_count, dtype: int64)

In [13]:
#Rename the raw ARFF feature names from Attr1-Attr64 to X1-X64 to match the official UCI feature notation.

feature_name_mapping = {
    f"Attr{i}": f"X{i}"
    for i in range(1, 65)
}

bankruptcy_1y = bankruptcy_1y.rename(
    columns=feature_name_mapping
)

bankruptcy_1y.columns[:10].tolist()


['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8', 'X9', 'X10']

In [14]:
# Count the number of unique values in each feature to detect constant or extremely low-variability columns

feature_unique_counts = (bankruptcy_1y.drop(columns="class").nunique(dropna=True).sort_values())

constant_features = feature_unique_counts[feature_unique_counts <= 1]
print(f"Constant features: {len(constant_features)}")
constant_features

Constant features: 0


Series([], dtype: int64)

In [15]:
feature_unique_counts.head(10)

X59    3254
X37    3260
X6     3539
X21    4420
X9     4823
X27    4985
X58    5115
X22    5144
X42    5158
X29    5214
dtype: int64

In [16]:
#Generate descriptive statistics for all financial features to identify scale differences, potentially extreme values and skewness

feature_summary = (bankruptcy_1y.drop(columns="class").describe().T)
feature_summary.head(10)

,count,mean,std,min,25%,50%,75%,max
X1,5907.0,-0.022347,6.163655,-4.638900e+02,0.003965,0.04667,0.117050,87.459
X2,5907.0,0.465086,5.751283,-4.308700e+02,0.255355,0.45175,0.661635,72.416
X3,5907.0,0.189155,1.177729,-7.206700e+01,0.043953,0.21944,0.418430,28.336
X4,5889.0,4.892476,91.434574,-4.031100e-01,1.093700,1.65170,2.931000,6845.800
X5,5899.0,19.406758,21529.322027,-1.076400e+06,-43.836500,0.49149,48.765000,1250100.000
X6,5907.0,0.022584,9.992080,-4.638900e+02,0.000000,0.00000,0.108725,543.250
X7,5907.0,-0.111951,9.057135,-5.174800e+02,0.005889,0.05650,0.136905,5.530
X8,5892.0,5.737741,102.355101,-3.735100e+00,0.481865,1.14930,2.771275,6868.500
X9,5909.0,1.588322,1.548390,-3.496000e+00,1.014800,1.13970,1.825900,65.607
X10,5907.0,0.545580,5.763742,-7.144400e+01,0.318940,0.52332,0.720805,339.850


In [17]:
# Rank features by their observed range to find financial ratios with extreme values

extreme_value_summary = feature_summary[["min", "25%", "50%", "75%", "max"]].copy()

extreme_value_summary["range"] = (extreme_value_summary["max"] - extreme_value_summary["min"])

extreme_value_summary.sort_values("range", ascending=False).head(15)

,min,25%,50%,75%,max,range
X15,-9.632400e+06,233.812500,872.165000,2255.825000,1341700.0,1.097410e+07
X55,-1.118500e+06,87.912250,1802.800000,7705.100000,4212200.0,5.330700e+06
X60,-1.244000e+01,5.248500,9.039500,17.319500,4818700.0,4.818712e+06
X32,-2.551000e+02,49.582250,81.126000,130.920000,4277200.0,4.277455e+06
X5,-1.076400e+06,-43.836500,0.491490,48.765000,1250100.0,2.326500e+06
X27,-1.581300e+05,0.075715,0.976060,4.135550,565940.0,7.240700e+05
X62,-2.365300e+02,44.646750,73.778500,118.720000,451380.0,4.516165e+05
X45,-3.037300e+03,0.017927,0.255145,0.799338,366030.0,3.690673e+05
X47,-1.865800e+01,19.627000,41.994000,73.868000,185610.0,1.856287e+05
X64,-3.726500e+00,2.147500,4.098300,9.204200,158180.0,1.581837e+05


In [18]:
#Identify all rows that belong to exact duplicate groups

duplicate_rows= bankruptcy_1y[bankruptcy_1y.duplicated(keep=False)].copy()

print(f"Rows involved in duplicate groups: {len(duplicate_rows):,}")
print(f"Redundant duplicate copies: {bankruptcy_1y.duplicated().sum():,}")

duplicate_rows.head(10)

Rows involved in duplicate groups: 120
Redundant duplicate copies: 60


,X1,X2,X3,X4,X5,X6,X7,X8,X9,X10,...,X56,X57,X58,X59,X60,X61,X62,X63,X64,class
102,0.174410,0.253050,0.494680,3.6883,86.14400,0.418790,0.219380,2.814600,1.16100,0.712230,...,0.138670,0.244880,0.86133,0.096938,8.4373,3.9381,47.262,7.7229,4.4228,0
103,0.174410,0.253050,0.494680,3.6883,86.14400,0.418790,0.219380,2.814600,1.16100,0.712230,...,0.138670,0.244880,0.86133,0.096938,8.4373,3.9381,47.262,7.7229,4.4228,0
319,-0.122210,0.980110,0.082083,1.1265,-7.81270,-0.306010,-0.102520,0.015187,0.96415,0.014885,...,-0.037185,-8.210400,1.03720,22.260000,16.9230,3.3988,116.450,3.1343,7.5552,0
320,-0.122210,0.980110,0.082083,1.1265,-7.81270,-0.306010,-0.102520,0.015187,0.96415,0.014885,...,-0.037185,-8.210400,1.03720,22.260000,16.9230,3.3988,116.450,3.1343,7.5552,0
366,0.527900,0.377340,0.517500,2.4936,79.87700,0.000000,0.527900,1.650100,1.55930,0.622660,...,0.137570,0.847820,0.86847,0.000000,6.8745,9.5438,81.108,4.5002,11.4650,0
410,0.059905,0.136320,0.184740,2.3747,0.17533,0.242290,0.074955,5.164700,1.10900,0.704070,...,0.098265,0.085083,0.90173,0.002754,9.0463,7.0221,55.945,6.5242,1.2877,0
463,0.000266,0.650370,0.063386,1.1493,-14.47500,0.006137,0.002881,0.504290,1.03920,0.327980,...,0.037686,0.000812,0.96231,0.688560,25.0940,4.9929,95.664,3.8154,3.1632,0
474,0.080963,0.429560,0.124900,1.6840,-6.08790,0.431180,0.101440,1.328000,0.73535,0.570440,...,0.249190,0.000000,0.76366,0.432930,6.3624,4.2257,90.637,4.0271,1.0619,0
475,0.080963,0.429560,0.124900,1.6840,-6.08790,0.431180,0.101440,1.328000,0.73535,0.570440,...,0.249190,0.000000,0.76366,0.432930,6.3624,4.2257,90.637,4.0271,1.0619,0
484,0.118140,0.068266,0.638710,10.3560,230.86000,0.279580,0.146840,13.119000,1.14160,0.895590,...,0.124030,0.131910,0.87597,0.000000,13.9230,7.2895,26.994,13.5210,3.1501,0


In [19]:
# Check whether duplicate observations are concentrated in one target class. This helps determine whether duplicates could distort the class distribution.

duplicate_target_distribution = (duplicate_rows["class"].value_counts().rename_axis("bankrupt").to_frame("count"))

duplicate_target_distribution["percentage"] = (duplicate_target_distribution["count"] / len(duplicate_rows) * 100).round(2)

duplicate_target_distribution

,count,percentage
bankrupt,,
0,116,96.67
1,4,3.33


In [20]:
#Check whether identical financial feature vectors appear with different targets.

feature_columns = bankruptcy_1y.columns.drop("class").tolist()

feature_duplicate_rows = bankruptcy_1y[bankruptcy_1y.duplicated(subset=feature_columns, keep=False)].copy()

conflicting_feature_groups = (feature_duplicate_rows.groupby(feature_columns, dropna=False)["class"].nunique())

conflicting_feature_groups = conflicting_feature_groups[conflicting_feature_groups > 1]

print(f"Rows sharing identical financial features: "f"{len(feature_duplicate_rows):,}")
print(f"Feature groups with conflicting target labels: "f"{len(conflicting_feature_groups):,}")

Rows sharing identical financial features: 120
Feature groups with conflicting target labels: 0


In [21]:
# Summarize the overall missing data problem

feature_data = bankruptcy_1y.drop(columns="class")

features_with_missing = feature_data.isna().any().sum()
total_missing_values = feature_data.isna().sum().sum()
rows_with_missing = feature_data.isna().any(axis=1).sum()

print(f"Features with missing values: {features_with_missing} / {feature_data.shape[1]}")
print(f"Total missing values: {total_missing_values:,}")
print(f"Rows with at least one missing value: {rows_with_missing:,}")
print(f"Rows with missing values (%): " f"{rows_with_missing / len(feature_data) * 100:.2f}%")

Features with missing values: 49 / 64
Total missing values: 4,666
Rows with at least one missing value: 2,879
Rows with missing values (%): 48.71%


In [22]:
#Rank features by missing value percentage

missing_summary = pd.DataFrame({
    "missing_count": feature_data.isna().sum(),
    "missing_percentage": (
        feature_data.isna().mean() * 100
    ).round(2)
})

missing_summary = (missing_summary.query("missing_count > 0").sort_values("missing_percentage", ascending=False))

missing_summary.head(15)

,missing_count,missing_percentage
X37,2548,43.11
X27,391,6.62
X60,268,4.53
X45,268,4.53
X24,135,2.28
X28,107,1.81
X64,107,1.81
X53,107,1.81
X54,107,1.81
X21,103,1.74


### Initial Data Quality Findings

- The dataset contains some duplicate observations that require further investigation before modeling
- No positive or negative infinite values
- Several financial ratios exhibit extremely wide ranges and substantial differences between their quartiles and observed min max values
- Missingness and extreme values will be investigated further during EDA and
  modeling. Extreme financial observations are retained rather than
  automatically removed, while missing-value handling is determined by the
  requirements of each model.

In [23]:
# Create a modeling copy of the dataset and remove redundant exact duplicates.

bankruptcy_1y_clean = (
    bankruptcy_1y
    .drop_duplicates()
    .reset_index(drop=True)
)


print(f"Original rows: {len(bankruptcy_1y):,}")
print(f"Rows after duplicate removal: {len(bankruptcy_1y_clean):,}")
print(f"Rows removed: {len(bankruptcy_1y) - len(bankruptcy_1y_clean):,}")

Original rows: 5,910
Rows after duplicate removal: 5,850
Rows removed: 60


In [24]:
# Check the duplicate numbers

remaining_duplicates = bankruptcy_1y_clean.duplicated().sum()

print(f"Remaining duplicate rows: {remaining_duplicates}")

Remaining duplicate rows: 0


In [25]:
# Recalculate the target distribution after removal

clean_target_distribution = (
    bankruptcy_1y_clean["class"]
    .value_counts()
    .rename_axis("bankrupt")
    .to_frame("count")
)

clean_target_distribution["percentage"] = (clean_target_distribution["count"] / len(bankruptcy_1y_clean) * 100).round(2)

clean_target_distribution

,count,percentage
bankrupt,,
0,5442,93.03
1,408,6.97


In [26]:
# Create a feature dictionary by using the official UCI definitions

feature_descriptions = {
    "X1": "net profit / total assets",
    "X2": "total liabilities / total assets",
    "X3": "working capital / total assets",
    "X4": "current assets / short-term liabilities",
    "X5": "[(cash + short-term securities + receivables - short-term liabilities) / (operating expenses - depreciation)] * 365",
    "X6": "retained earnings / total assets",
    "X7": "EBIT / total assets",
    "X8": "book value of equity / total liabilities",
    "X9": "sales / total assets",
    "X10": "equity / total assets",
    "X11": "(gross profit + extraordinary items + financial expenses) / total assets",
    "X12": "gross profit / short-term liabilities",
    "X13": "(gross profit + depreciation) / sales",
    "X14": "(gross profit + interest) / total assets",
    "X15": "(total liabilities * 365) / (gross profit + depreciation)",
    "X16": "(gross profit + depreciation) / total liabilities",
    "X17": "total assets / total liabilities",
    "X18": "gross profit / total assets",
    "X19": "gross profit / sales",
    "X20": "(inventory * 365) / sales",
    "X21": "sales (n) / sales (n-1)",
    "X22": "profit on operating activities / total assets",
    "X23": "net profit / sales",
    "X24": "gross profit (in 3 years) / total assets",
    "X25": "(equity - share capital) / total assets",
    "X26": "(net profit + depreciation) / total liabilities",
    "X27": "profit on operating activities / financial expenses",
    "X28": "working capital / fixed assets",
    "X29": "logarithm of total assets",
    "X30": "(total liabilities - cash) / sales",
    "X31": "(gross profit + interest) / sales",
    "X32": "(current liabilities * 365) / cost of products sold",
    "X33": "operating expenses / short-term liabilities",
    "X34": "operating expenses / total liabilities",
    "X35": "profit on sales / total assets",
    "X36": "total sales / total assets",
    "X37": "(current assets - inventories) / long-term liabilities",
    "X38": "constant capital / total assets",
    "X39": "profit on sales / sales",
    "X40": "(current assets - inventory - receivables) / short-term liabilities",
    "X41": "total liabilities / ((profit on operating activities + depreciation) * (12 / 365))",
    "X42": "profit on operating activities / sales",
    "X43": "rotation receivables + inventory turnover in days",
    "X44": "(receivables * 365) / sales",
    "X45": "net profit / inventory",
    "X46": "(current assets - inventory) / short-term liabilities",
    "X47": "(inventory * 365) / cost of products sold",
    "X48": "EBITDA / total assets",
    "X49": "EBITDA / sales",
    "X50": "current assets / total liabilities",
    "X51": "short-term liabilities / total assets",
    "X52": "(short-term liabilities * 365) / cost of products sold",
    "X53": "equity / fixed assets",
    "X54": "constant capital / fixed assets",
    "X55": "working capital",
    "X56": "(sales - cost of products sold) / sales",
    "X57": "(current assets - inventory - short-term liabilities) / (sales - gross profit - depreciation)",
    "X58": "total costs / total sales",
    "X59": "long-term liabilities / equity",
    "X60": "sales / inventory",
    "X61": "sales / receivables",
    "X62": "(short-term liabilities * 365) / sales",
    "X63": "sales / short-term liabilities",
    "X64": "sales / fixed assets",
}

In [27]:
#Convert the feature dictionary into a DataFrame

feature_dictionary = pd.DataFrame(feature_descriptions.items(), columns=["feature", "description"])

feature_dictionary.head(10)

,feature,description
0,X1,net profit / total assets
1,X2,total liabilities / total assets
2,X3,working capital / total assets
3,X4,current assets / short-term liabilities
4,X5,[(cash + short-term securities + receivables -...
5,X6,retained earnings / total assets
6,X7,EBIT / total assets
7,X8,book value of equity / total liabilities
8,X9,sales / total assets
9,X10,equity / total assets


In [28]:
#Add a financial definitions to the missing value summary to make affected features easier to interpret.

missing_summary_with_description = (
    missing_summary
    .reset_index()
    .rename(columns={"index": "feature"})
)

missing_summary_with_description["description"] = (missing_summary_with_description["feature"].map(feature_descriptions))

missing_summary_with_description.head(15)

,feature,missing_count,missing_percentage,description
0,X37,2548,43.11,(current assets - inventories) / long-term lia...
1,X27,391,6.62,profit on operating activities / financial exp...
2,X60,268,4.53,sales / inventory
3,X45,268,4.53,net profit / inventory
4,X24,135,2.28,gross profit (in 3 years) / total assets
5,X28,107,1.81,working capital / fixed assets
6,X64,107,1.81,sales / fixed assets
7,X53,107,1.81,equity / fixed assets
8,X54,107,1.81,constant capital / fixed assets
9,X21,103,1.74,sales (n) / sales (n-1)


## Data Understanding Summary

The assesments of the 1 year bankruptcy dataset shows several important characteristics:

- Bankruptcy is the minority class, representing approximately 7% of the observations.
- 60 redundant exact duplicate observations were identified and removed from the modeling copy.
- No conflicting target labels were found among identical financial feature vectors.
- No infinite numerical values or constant features were detected.
- 49 of the 64 financial features contain at least one missing value in the original dataset.
- 2,879 companies in the original dataset contain at least one missing financial value.
- X37 has the highest missing-data rate at approximately 43%.
- Several financial ratios contain extreme values and highly skewed ranges.

These findings guide the exploratory analysis and the preprocessing and modeling decisions used in the subsequent notebooks.